# lib and load

In [76]:
# Import core analytical and visualization libraries
import pandas as pd
import numpy as np
import os
import warnings

from datetime import timedelta
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import lightgbm as lgb
import xgboost as xgb

# Ignore warning messages for cleaner notebook output
warnings.filterwarnings('ignore')

In [77]:
DATA_PATH = '../dataset/04_after_fe/'
print("Loading data...")

df_daily = pd.read_parquet(DATA_PATH + 'train_data.parquet')
df_sample = pd.read_csv('../dataset/01_raw/sample_submission.csv')

print("Load data successfully")

Loading data...
Load data successfully


# info

In [78]:
df_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Columns: 103 entries, Date to cog_lag_365
dtypes: datetime64[ns](1), float64(88), int32(3), int64(11)
memory usage: 3.0 MB


# model

In [ ]:
class DatathonForecaster:
    def __init__(self):
        self.models_rev = {}
        self.models_cog = {}
        self.weights_rev = {}
        self.weights_cog = {}
        self.feature_cols = []
        
        
    def build_historical_profile(self, df_train):
        """
        Cập nhật: Chỉ giữ lại các biến Lag của Category.
        Loại bỏ Size và Color để giảm nhiễu và tránh lan truyền sai số.
        """
        # CHỈ lấy các biến Category (Sát với Revenue nhất)
        self.sub_features = [
            col for col in df_train.columns 
            if ('qty_lag' in col or 'rank_lag' in col) 
            and ('cat_' in col) # Chỉ giữ lại Category
            and not ('size_' in col) # Loại bỏ Size
            and not ('color_' in col) # Loại bỏ Color
        ]
        
        if not self.sub_features:
            # Nếu không có cat_, lấy tạm các biến lag chung (không theo phân loại chi tiết)
            self.sub_features = [col for col in df_train.columns if 'qty_lag1' in col and '_' not in col.replace('qty_lag1','')]
            
        group_keys = ['day_of_week', 'category_promotion', 'chanel_promotion']
        self.historical_profile = df_train.groupby(group_keys)[self.sub_features].mean().reset_index()
        print(f"[+] Đã tối ưu hóa: Chỉ giữ {len(self.sub_features)} biến Category quan trọng nhất.")
        

    # ==========================================
    # PHASE 1: FEATURE ENGINEERING
    # ==========================================
    def apply_static_features(self, df):
        """
        Tái tạo chính xác logic Static Features (từ ảnh của bạn).
        Hàm này được dùng chung cho cả tập Train lúc đầu và từng ngày Test lúc Recursive.
        """
        df = df.copy()
        df['month_day'] = df['Date'].dt.strftime('%m-%d')
        df['day'] = df['Date'].dt.day
        df['month'] = df['Date'].dt.month
        df['year'] = df['Date'].dt.year
        df['day_of_week'] = df['Date'].dt.dayofweek
        
        df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
        df['is_first_7_days'] = (df['day'] <= 7).astype(int)
        df['is_last_7_days'] = (df['day'] > (df['Date'].dt.days_in_month - 7)).astype(int)
        df['quarter'] = df['month'] % 12 // 3 + 1
        
        # Promotions
        df['is_Spring_promotion'] = df['month_day'].between('03-18', '04-17').astype(int)
        df['is_MidYear_promotion'] = df['month_day'].between('06-23', '07-22').astype(int)
        df['is_FallLauch_promotion'] = ((df['year'] % 4 == 1) & (df['month_day'] == '10-02')).astype(int)
        df['is_YearEnd_promotion'] = (df['month_day'].between('11-18', '12-31') | df['month_day'].between('01-01', '01-02')).astype(int)
        df['is_Rural_promotion'] = ((df['year'] % 2 == 1) & df['month_day'].between('01-30', '03-01')).astype(int)
        df['is_Urban_promotion'] = ((df['year'] % 2 == 1) & df['month_day'].between('07-30', '09-02')).astype(int)
        
        # Default categories & channels
        df['category_promotion'] = -1 
        df['chanel_promotion'] = -1
        
        # Decode category
        df.loc[(df['is_Spring_promotion']==1) | (df['is_MidYear_promotion']==1) | (df['is_YearEnd_promotion']==1), 'category_promotion'] = 0
        df.loc[df['is_Rural_promotion']==1, 'category_promotion'] = 1
        df.loc[df['is_Urban_promotion']==1, 'category_promotion'] = 2
        
        # Decode channel
        df.loc[(df['is_Spring_promotion']==1) | (df['is_FallLauch_promotion']==1), 'chanel_promotion'] = 4
        df.loc[df['is_Rural_promotion']==1, 'chanel_promotion'] = 1
        df.loc[df['is_Urban_promotion']==1, 'chanel_promotion'] = 2
        df.loc[df['is_MidYear_promotion']==1, 'chanel_promotion'] = 3
        df.loc[df['is_YearEnd_promotion']==1, 'chanel_promotion'] = 0
        
        df = df.drop(columns=['month_day', 'day', 'year'], errors='ignore')
        return df

    def create_features(self, df):
        """
        Đảm bảo KHÔNG RÒ RỈ DỮ LIỆU. Dynamic Features phải được tính từ quá khứ.
        """
        df = df.sort_values('Date').reset_index(drop=True)
        
        # 1. Gọi hàm tạo Static Features
        df = self.apply_static_features(df)
        
        # 2. Dynamic Features: Lags (t-1, t-7, t-30)
        lags = [1, 7, 365]
        for lag in lags:
            df[f'rev_lag_{lag}'] = df['Revenue'].shift(lag)
            df[f'cog_lag_{lag}'] = df['COGS'].shift(lag)
            
        # 3. Rolling Features (Mean, Std)
        # BẮT BUỘC shift(1) trước khi rolling để mô hình không nhìn thấy data của ngày hôm nay
        windows = [7, 365]
        for w in windows:
            df[f'rev_roll_mean_{w}'] = df['Revenue'].shift(1).rolling(window=w).mean()
            df[f'rev_roll_std_{w}'] = df['Revenue'].shift(1).rolling(window=w).std()
            df[f'cog_roll_mean_{w}'] = df['COGS'].shift(1).rolling(window=w).mean()
            df[f'cog_roll_std_{w}'] = df['COGS'].shift(1).rolling(window=w).std()
            
        # 4. Clean up: Drop các cột raw metrics của ngày hiện tại (nếu có tồn tại do join nhầm)
        # Bất kỳ cột nào liên quan đến qty, rank mà không có chữ 'lag' đều là leakage
        leak_cols = [c for c in df.columns if ('qty' in c or 'rank' in c) and ('lag' not in c)]
        df = df.drop(columns=leak_cols, errors='ignore')
        
        return df

    # ==========================================
    # PHASE 2: TRAINING PIPELINE
    # ==========================================
    def get_base_models(self):
        return {
            'LGBM': (lgb.LGBMRegressor(random_state=42, verbose=-1),
                     {'learning_rate': [0.01, 0.05], 'n_estimators': [100, 300], 'num_leaves': [31, 63]}),
            'XGB': (xgb.XGBRegressor(random_state=42, objective='reg:squarederror'),
                    {'learning_rate': [0.01, 0.05], 'n_estimators': [100, 300], 'max_depth': [3, 5]}),
            'RF': (RandomForestRegressor(random_state=42),
                   {'n_estimators': [100, 200], 'max_depth': [5, 10, None]})
        }

    def tune_global_parameters(self, df_train):
        print("\n[+] ĐANG KHỞI ĐỘNG CHIẾN DỊCH GLOBAL TUNING (100 Iterations)...")
        self.feature_cols = [c for c in df_train.columns if c not in ['Date', 'Revenue', 'COGS']]
        
        X_full = df_train[self.feature_cols].bfill().fillna(value=0)
        y_full_rev = df_train['Revenue']
        y_full_cog = df_train['COGS']
        
        # Dùng TimeSeriesSplit để tránh Leakage khi chia Fold tìm thông số
        tscv = TimeSeriesSplit(n_splits=3)
        base_models = self.get_base_models()
        
        best_params_rev = {}
        best_params_cog = {}
        
        for name, (model, params) in base_models.items():
            print(f" -> Đang Tuning cho {name}...")
            
            # Tuning cho Revenue (Chạy 100 lần thử nghiệm)
            search_rev = RandomizedSearchCV(
                model, params, n_iter=100, cv=tscv, 
                scoring='neg_mean_squared_error', random_state=42, n_jobs=-1
            )
            search_rev.fit(X_full, y_full_rev)
            best_params_rev[name] = search_rev.best_params_
            
            # Tuning cho COGS (Chạy 100 lần thử nghiệm)
            search_cog = RandomizedSearchCV(
                model, params, n_iter=100, cv=tscv, 
                scoring='neg_mean_squared_error', random_state=42, n_jobs=-1
            )
            search_cog.fit(X_full, y_full_cog)
            best_params_cog[name] = search_cog.best_params_
            
        print("[+] Hoàn tất Tuning! Đã chốt bộ thông số Global tốt nhất.")
        return best_params_rev, best_params_cog
    
    def calculate_ensemble_weights(self, rmse_dict):
        """
        Tính trọng số tỷ lệ nghịch với RMSE trung bình.
        Công thức: W_i = (1/RMSE_i) / Sum(1/RMSE_j)
        """
        avg_rmse = {name: np.mean(scores) for name, scores in rmse_dict.items()}
        inv_rmse = {name: 1.0 / score for name, score in avg_rmse.items() if score > 0}
        total_inv = sum(inv_rmse.values())
        return {name: val / total_inv for name, val in inv_rmse.items()}

    # ==========================================
    # HÀM TÍNH TOÁN TRỌNG SỐ ĐỘNG (THÔNG MINH HƠN 6:3:1)
    # ==========================================
    def calculate_dynamic_weights(self, mse_val_dict, mse_train_dict, target_name):
        """
        Logic tính tỷ lệ: 
        1. Kết hợp Val MSE và Train MSE (Tỷ trọng 7:3).
        2. Phạt mô hình có dấu hiệu Overfitting (Val MSE >> Train MSE).
        3. Bình phương sai số để khuếch đại tỷ lệ cho model dẫn đầu.
        """
        weights = {}
        adjusted_errors = {}
        
        print(f"\n[+] TÍNH TOÁN TỶ LỆ DỰ ĐOÁN CHO {target_name.upper()}:")
        
        for name in mse_val_dict.keys():
            val_mse = np.mean(mse_val_dict[name]) # Lấy trung bình MSE qua các Fold
            train_mse = mse_train_dict[name]      # MSE khi học full 10 năm
            
            # Hình phạt Overfit: Chỉ phạt khi Val cao hơn Train (Học vẹt)
            overfit_penalty = max(0, val_mse - train_mse)
            
            # Tính lỗi điều chỉnh
            adj_error = (0.7 * val_mse) + (0.3 * train_mse) + (0.5 * overfit_penalty)
            adjusted_errors[name] = adj_error
            
            print(f"   [{name}] Avg Val MSE: {val_mse:,.0f} | Full Train MSE: {train_mse:,.0f} | Overfit Penalty: {overfit_penalty:,.0f}")
            
        # Tính tỷ lệ nghịch đảo bình phương để tạo ra khoảng cách kiểu 60-30-10
        inverse_errors = {name: 1.0 / (error ** 2) for name, error in adjusted_errors.items()}
        total_inverse = sum(inverse_errors.values())
        
        # Chuyển đổi thành phần trăm (%)
        for name in inverse_errors.keys():
            weights[name] = inverse_errors[name] / total_inverse
            print(f"   => Tỷ lệ chốt cho {name}: {weights[name]*100:.2f}%")
            
        return weights

    def train_pipeline(self, df):
        """
        Thực thi Walk-Forward Validation chu kỳ 1.5 năm (18 tháng).
        Học từ 01/2013 -> Xếp hạng Model -> Refit toàn bộ đến 12/2022.
        """
        start_date = pd.to_datetime('2013-01-01')
        end_date = pd.to_datetime('2022-12-31')
        df_train = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)].reset_index(drop=True)
        
        # 1. GỌI HÀM TUNING ĐỂ TÌM THÔNG SỐ TRƯỚC
        best_params_rev, best_params_cog = self.tune_global_parameters(df_train)
        
        # 2. KHỞI TẠO MODEL VỚI THÔNG SỐ ĐÃ CHỐT
        base_models = self.get_base_models()
        models_rev = {name: base_models[name][0].set_params(**best_params_rev[name]) for name in base_models.keys()}
        models_cog = {name: base_models[name][0].set_params(**best_params_cog[name]) for name in base_models.keys()}

        # 3. VÒNG LẶP WALK-FORWARD (Chỉ để Test lấy MSE và tính tỷ lệ)
        step_months = 18 
        current_train_end = start_date + pd.DateOffset(months=step_months)
        
        mse_rev = {'LGBM': [], 'XGB': [], 'RF': []}
        mse_cog = {'LGBM': [], 'XGB': [], 'RF': []}
        
        fold = 1
        while current_train_end <= end_date:
            val_end = current_train_end + pd.DateOffset(months=step_months)
            if val_end > end_date: val_end = end_date + pd.DateOffset(days=1)
                
            print(f"\n--- WALK-FORWARD CHU KỲ {fold} | ĐOÁN: {current_train_end.date()} -> {(val_end - pd.Timedelta(days=1)).date()} ---")
            
            train_mask = (df_train['Date'] < current_train_end)
            val_mask = (df_train['Date'] >= current_train_end) & (df_train['Date'] < val_end)
            if not val_mask.any(): break
            
            X_train = df_train.loc[train_mask, self.feature_cols].bfill().fillna(value=0)
            X_val = df_train.loc[val_mask, self.feature_cols].bfill().fillna(value=0)
            y_train_rev, y_train_cog = df_train.loc[train_mask, 'Revenue'], df_train.loc[train_mask, 'COGS']
            y_val_rev, y_val_cog = df_train.loc[val_mask, 'Revenue'], df_train.loc[val_mask, 'COGS']

            # Lúc này Model CỨ THẾ MÀ FIT VÀ PREDICT, không cần search gì nữa -> Chạy cực nhanh!
            for name in models_rev.keys():
                # Test Revenue
                models_rev[name].fit(X_train, y_train_rev)
                preds_rev = models_rev[name].predict(X_val)
                
                # Tính Metrics
                mse_r = mean_squared_error(y_val_rev, preds_rev)
                rmse_r = np.sqrt(mse_r)
                r2_r = r2_score(y_val_rev, preds_rev)
                rma_r = mean_absolute_error(y_val_rev, preds_rev) 
                
                mse_rev[name].append(mean_squared_error(y_val_rev, preds_rev))
                print(f"   [Rev - {name:<4}] RMSE: {rmse_r:,.2f} | R^2: {r2_r:5.4f} | MAE(RMA): {rma_r:,.2f}")

                # Test COGS
                models_cog[name].fit(X_train, y_train_cog)
                preds_cog = models_cog[name].predict(X_val)
                
                # Tính Metrics
                mse_c = mean_squared_error(y_val_cog, preds_cog)
                rmse_c = np.sqrt(mse_c)
                r2_c = r2_score(y_val_cog, preds_cog)
                rma_c = mean_absolute_error(y_val_cog, preds_cog)

                mse_cog[name].append(mean_squared_error(y_val_cog, preds_cog))
                print(f"   [Cog - {name:<4}] RMSE: {rmse_c:,.2f} | R^2: {r2_c:5.4f} | MAE(RMA): {rma_c:,.2f}")

            current_train_end = val_end
            fold += 1

        # 4. FINAL REFIT TRÊN FULL 10 NĂM ĐỂ SẴN SÀNG INFERENCE
        print("\n[+] Đang tiến hành Final Refit trên toàn bộ dữ liệu Train (2013 - 2022)...")
        X_full = df_train[self.feature_cols].bfill().fillna(value=0)
        y_full_rev, y_full_cog = df_train['Revenue'], df_train['COGS']
        
        mse_train_rev = {}
        mse_train_cog = {}
        
        print("\n--- KẾT QUẢ ĐÁNH GIÁ TRÊN TOÀN BỘ TẬP HỌC (FULL TRAIN) ---")
        for name in models_rev.keys():
            # Fit Full
            models_rev[name].fit(X_full, y_full_rev)
            models_cog[name].fit(X_full, y_full_cog)
    
            # Predict Full
            preds_train_rev = models_rev[name].predict(X_full)
            preds_train_cog = models_cog[name].predict(X_full)

            # Tính Metrics Revenue Full
            mse_train_r = mean_squared_error(y_full_rev, preds_train_rev)
            rmse_train_r = np.sqrt(mse_train_r)
            r2_train_r = r2_score(y_full_rev, preds_train_rev)
            rma_train_r = mean_absolute_error(y_full_rev, preds_train_rev)
            
            # Tính Metrics COGS Full
            mse_train_c = mean_squared_error(y_full_cog, preds_train_cog)
            rmse_train_c = np.sqrt(mse_train_c)
            r2_train_c = r2_score(y_full_cog, preds_train_cog)
            rma_train_c = mean_absolute_error(y_full_cog, preds_train_cog)
            
            mse_train_rev[name] = mse_train_r
            mse_train_cog[name] = mse_train_c

            # Lưu model vào self để đưa xuống hàm predict_future
            self.models_rev[name] = models_rev[name]
            self.models_cog[name] = models_cog[name]

            print(f"   [FULL Rev - {name:<4}] RMSE: {rmse_train_r:,.2f} | R^2: {r2_train_r:5.4f} | MAE(RMA): {rma_train_r:,.2f}")
            print(f"   [FULL Cog - {name:<4}] RMSE: {rmse_train_c:,.2f} | R^2: {r2_train_c:5.4f} | MAE(RMA): {rma_train_c:,.2f}")

        # 5. GỌI HÀM TÍNH TỶ LỆ TRỌNG SỐ ĐỘNG (Dựa trên Val MSE và Train MSE)
        self.weights_rev = self.calculate_dynamic_weights(mse_rev, mse_train_rev, target_name="Revenue")
        self.weights_cog = self.calculate_dynamic_weights(mse_cog, mse_train_cog, target_name="COGS")
        
        print("\n[+] Training Pipeline Hoàn Tất! Đã sẵn sàng cho đệ quy.")
        

    # ==========================================
    # PHASE 3: INFERENCE STRATEGY (RECURSIVE)
    # ==========================================
    def predict_future(self, test_dates, df_historical):
        print("\nBắt đầu Recursive Inference với Historical Imputation...")
        
        # Build profile từ tập historical ban đầu (tập Train)
        self.build_historical_profile(df_historical)
        
        hist_df = df_historical.copy().sort_values('Date').tail(40).reset_index(drop=True)
        predictions = []
        
        for current_date in test_dates:
            # 1. Tạo row cho ngày hiện tại (chỉ có Date)
            new_row = pd.DataFrame({'Date': [current_date], 'Revenue': [np.nan], 'COGS': [np.nan]})
            
            # --- LOGIC CỦA BẠN: ĐIỀN DATA CHO NGÀY HÔM QUA ---
            # Trước khi concat ngày hôm nay vào, ta điền các biến sub-features cho ngày cuối cùng của hist_df (tức là ngày hôm qua)
            if self.historical_profile is not None and not hist_df.empty:
                yesterday_idx = hist_df.index[-1]
                
                # Trích xuất đặc điểm của ngày hôm qua
                y_dow = hist_df.loc[yesterday_idx, 'day_of_week']
                y_cat_pro = hist_df.loc[yesterday_idx, 'category_promotion']
                y_chan_pro = hist_df.loc[yesterday_idx, 'chanel_promotion']
                
                # Tìm "các ngày giống ngày hôm qua" trong Profile
                match = self.historical_profile[
                    (self.historical_profile['day_of_week'] == y_dow) &
                    (self.historical_profile['category_promotion'] == y_cat_pro) &
                    (self.historical_profile['chanel_promotion'] == y_chan_pro)
                ]
                
                if not match.empty:
                    # Điền giá trị trung bình vào ngày hôm qua
                    for col in self.sub_features:
                        hist_df.loc[yesterday_idx, col] = match[col].values[0]
                else:
                    # Fallback: Lấy trung bình toàn bộ nếu không tìm thấy match chính xác
                    for col in self.sub_features:
                        hist_df.loc[yesterday_idx, col] = self.historical_profile[col].mean()
            # -------------------------------------------------
            
            # 2. Concat row ngày hôm nay vào
            hist_df = pd.concat([hist_df, new_row], ignore_index=True)
            
            # 3. Tính toán lại Features (Lúc này hàm shift(1) sẽ lấy được các giá trị trung bình ta vừa điền ở trên)
            current_features_df = self.create_features(hist_df)
            
            # 4. Lấy vector ngày hôm nay đưa vào dự báo
            X_test = current_features_df.iloc[-1:][self.feature_cols].fillna(value=0)
            
            # 5. Ensemble Predict & Ràng buộc COGS < Revenue
            pred_rev = sum(self.models_rev[name].predict(X_test)[0] * weight for name, weight in self.weights_rev.items())
            pred_cog = sum(self.models_cog[name].predict(X_test)[0] * weight for name, weight in self.weights_cog.items())
            if pred_cog >= pred_rev: pred_cog = pred_rev * 0.90
            
            # 6. Cập nhật Revenue/COGS để chạy lag tiếp theo
            hist_df.loc[hist_df.index[-1], 'Revenue'] = pred_rev
            hist_df.loc[hist_df.index[-1], 'COGS'] = pred_cog
            
            predictions.append({'Date': current_date, 'Revenue': pred_rev, 'COGS': pred_cog})
            hist_df = hist_df.tail(40).reset_index(drop=True)
            
        return pd.DataFrame(predictions)

# ==========================================
# THỰC THI PIPELINE
# ==========================================
if __name__ == "__main__":
    # 1. Load Data
    # Giả định df_raw đã được đọc từ 'train_data.parquet'
    # df_raw = pd.read_parquet('train_data.parquet')
    
    # 2. Khởi tạo Pipeline
    pipeline = DatathonForecaster()
    
    # 3. Feature Engineering trên tập Train
    df_featured = pipeline.create_features(df_daily)
    
    # 4. Training (Học & Tính trọng số theo chu kỳ 1.5 năm)
    pipeline.train_pipeline(df_featured)
    
    # 5. Recursive Inference cho tập Test (1.5 năm: 01/01/2023 - 01/07/2024)
    test_dates = pd.date_range(start='2023-01-01', end='2024-07-01', freq='D')
    final_submission = pipeline.predict_future(test_dates, df_historical=df_featured)
    
    final_submission.to_csv('submission.csv', index=False)
    print("Đã lưu file submission.csv thành công!")
    pass


[+] ĐANG KHỞI ĐỘNG CHIẾN DỊCH GLOBAL TUNING (100 Iterations)...
 -> Đang Tuning cho LGBM...
 -> Đang Tuning cho XGB...
 -> Đang Tuning cho RF...
[+] Hoàn tất Tuning! Đã chốt bộ thông số Global tốt nhất.

--- WALK-FORWARD CHU KỲ 1 | ĐOÁN: 2014-07-01 -> 2015-12-31 ---
   [Rev - LGBM] RMSE: 1,217,169.64 | R^2: 0.7688 | MAE(RMA): 833,974.91
   [Cog - LGBM] RMSE: 1,035,151.29 | R^2: 0.7681 | MAE(RMA): 704,744.91
   [Rev - XGB ] RMSE: 1,206,351.26 | R^2: 0.7729 | MAE(RMA): 848,683.39
   [Cog - XGB ] RMSE: 992,275.21 | R^2: 0.7870 | MAE(RMA): 696,328.29
   [Rev - RF  ] RMSE: 1,252,752.29 | R^2: 0.7551 | MAE(RMA): 844,777.23
   [Cog - RF  ] RMSE: 1,072,307.07 | R^2: 0.7512 | MAE(RMA): 716,472.77

--- WALK-FORWARD CHU KỲ 2 | ĐOÁN: 2016-01-01 -> 2017-06-30 ---
   [Rev - LGBM] RMSE: 1,473,575.35 | R^2: 0.7973 | MAE(RMA): 983,836.70
   [Cog - LGBM] RMSE: 1,280,580.33 | R^2: 0.7835 | MAE(RMA): 864,085.87
   [Rev - XGB ] RMSE: 1,605,972.00 | R^2: 0.7592 | MAE(RMA): 1,050,859.14
   [Cog - XGB ] RMSE

In [80]:
final_submission.info()
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 548 entries, 0 to 547
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     548 non-null    datetime64[ns]
 1   Revenue  548 non-null    float64       
 2   COGS     548 non-null    float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 13.0 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 548 entries, 0 to 547
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     548 non-null    object 
 1   Revenue  548 non-null    float64
 2   COGS     548 non-null    float64
dtypes: float64(2), object(1)
memory usage: 13.0+ KB
